In [102]:
import numpy as np

def ConvertToLatLon(dX, dY, lat):
    earthRadius = 6.371e6
    latRadian = np.radians(lat)
    dLatRadian = dY / earthRadius
    dLonRadian = dX / (earthRadius * np.cos(latRadian))
    return np.degrees(dLatRadian), np.degrees(dLonRadian)


In [103]:
latCenter, lonCenter = 29.67, -95.059
dx = dy = 1000.0  # meters

dLat, dLon = ConvertToLatLon(dx, dy, latCenter)
print(dLat, dLon) # 0.008993216059187304 0.010350225694853669

# RESULT #*#*
# ==> dLat and dLon is about 0.01 #*#*

0.008993216059187304 0.010350225694853669


In [109]:
nLat, nLon = 500, 500

half_nLat = nLat / 2
half_nLon = nLon / 2

# start and end values (symmetrical around center)
startlat = latCenter - half_nLat * dLat
endlat   = latCenter + half_nLat * dLat
startlon = lonCenter - half_nLon * dLon
endlon   = lonCenter + half_nLon * dLon

# RESULT #*#*
# startlat, endlat = (27.421695985203176, 31.918304014796828) ==> (27, 33)
# startlon, endlon = (-97.64655642371342, -92.47144357628657) ==> (-98, -92)
# nLat, nLon = 500, 500

In [ ]:
#old convert_mpas configuration
# startlat=15
# endlat=45
# startlon=-120
# endlon=-70

In [120]:
######################
#RESULTS AFTER RUNNING
######################

In [121]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [122]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
from CLASSES_Directories import DirectoryManager_Class

In [123]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "AreaAverages"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

 DirectoryManager_Class Summary
 Main Directory:           /glade/u/home/aroseman/Projects/Regional-MPAS-Project
 Main Scratch Directory:   /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project
 Scratch Directory:        /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0
 Main Output Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT
 Main Output Plotting Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/PLOTTING
 Main Code Directory:      /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles
 Current Code Directory:   /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/DataAnalysis/MPAS_Model_Data/InitialFigures



In [124]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "MOIST"):
        SimulationTime = ("2022-06-30","2022-07-03")
    return SimulationTime

In [128]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "MOIST"):
        SimulationTime = ("2022-06-30","2022-07-03")
    return SimulationTime

# RunType = ("TRACER","MOIST","NSSL")
RunType = ("TRACER","MOIST","TEMPO")
SimulationTime = GetSimulationTime(RunType)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

Found 289/289 files matching expected times.
Opened history file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_TEMPO/history_cartesian/history.2022-06-30_00.00.00.latlon.nc
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_TEMPO/diag_cartesian/diag.2022-06-30_00.00.00.latlon.nc

=== MPAS Structured (lat-lon) Model Data Summary ===
 Region:         TRACER
 Case:           MOIST
 Microphysics:   TEMPO
 Resolution:     20-1km
 Time Step:      15mins
 Time Range:     2022-06-30 to 2022-07-03
 Coordinates:    ['latitude', 'longitude', 'nVertLevels', 'nVertLevelsP1']
 # History Files:289
 # Diag Files:   289
 # Time Steps:   289
 Data Directory: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_TEMPO
 Static File:    TRACER_regional5250_scaled3_x20.835586.static.latlon.nc



In [129]:
#testing dLat dLon
hi = ModelData.GetDataTimestep(t=0)
a = np.mean(np.diff(hi['longitude'])*(np.pi/180)*6371e3*np.cos(hi['latitude'].data*np.pi/180)[0:-1])
b = np.mean(np.diff(hi['latitude'])*(np.pi/180)*6371e3)
print(a,b)

Opened history file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_TEMPO/history_cartesian/history.2022-06-30_00.00.00.latlon.nc
1155.1168 1334.3397
